# **Membaca Dataset Utama**

In [2]:
import pandas as pd

# 1. Memuat dataset utama
file_name = 'Dataset 2D Twin Final.xlsx'
df = pd.read_excel(file_name, sheet_name='intent_dataset')

# 2. Menampilkan 10 data teratas (df.head) untuk melihat struktur kolom dan isinya
print("="*30)
print("1. TAMPILAN 10 DATA TERATAS (df.head):")
print("="*30)
display(df.head(10))

# 3. Mengecek informasi umum terkait dataset (df.info) seperti tipe data dan missing value
print("\n" + "="*30)
print("2. INFORMASI STRUKTUR DATASET (df.info):")
print("="*30)
df.info()

# 4. Mengecek dimensi data (jumlah baris dan kolom)
print("\n" + "="*30)
print("3. DIMENSI DATASET (Baris, Kolom):")
print("="*30)
print(f"Dataset ini memiliki {df.shape[0]} baris dan {df.shape[1]} kolom.")

# 5. Menghitung distribusi jumlah data per kelas Intent
print("\n" + "="*30)
print("4. DISTRIBUSI JUMLAH DATA PER INTENT:")
print("="*30)
print(df['intent'].value_counts())

1. TAMPILAN 10 DATA TERATAS (df.head):


,text,intent,item,source,target,quantity,Unnamed: 6,Unnamed: 7
0,hilangkan 4 komputer dari ruang a,remove_stock,komputer,ruang a,ruang server,4,NaN,NaN
1,lacak komputer,search_item,komputer,lantai 1,lobby,12,NaN,Penjelasan Mengenai Kolom Kosong atau bernilai...
2,lokasi proyektor dimana?,search_item,proyektor,basement,lantai 2,26,NaN,NaN
3,hapus 46 kabel dari ruang b,remove_stock,kabel,ruang b,ruang arsip,46,NaN,NaN
4,tempat laptop dimana?,search_item,laptop,ruang tamu,ruang keuangan,29,NaN,NaN
5,pindahkan 44 kamera dari lantai 3 ke ruang arsip,move_item,kamera,lantai 3,ruang arsip,44,NaN,NaN
6,berapa stok di ruang meeting?,check_stock,meja,ruang meeting,lantai 1,10,NaN,NaN
7,add 41 scanner ke ruang server,add_stock,scanner,ruang b,ruang server,41,NaN,NaN
8,hilangkan 40 flashdisk dari ruang it,remove_stock,flashdisk,ruang it,ruang tamu,40,NaN,NaN
9,pindah 50 monitor dari lantai 2 ke lantai 2,move_item,monitor,lantai 2,lantai 2,50,NaN,NaN



2. INFORMASI STRUKTUR DATASET (df.info):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4480 entries, 0 to 4479
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   text        4480 non-null   object 
 1   intent      4480 non-null   object 
 2   item        3900 non-null   object 
 3   source      3946 non-null   object 
 4   target      3921 non-null   object 
 5   quantity    4480 non-null   int64  
 6   Unnamed: 6  0 non-null      float64
 7   Unnamed: 7  1 non-null      object 
dtypes: float64(1), int64(1), object(6)
memory usage: 280.1+ KB

3. DIMENSI DATASET (Baris, Kolom):
Dataset ini memiliki 4480 baris dan 8 kolom.

4. DISTRIBUSI JUMLAH DATA PER INTENT:
intent
move_item           1108
remove_stock        1006
add_stock            968
search_item          365
check_stock          313
low_stock             80
room_information      80
zone_information      80
count_zone            80
count_item            80
m

# **Label Encoding & Data Splitting**

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. Daftarkan 12 Intent resmi
intent_resmi = [
    'search_item', 'check_stock', 'low_stock', 'room_information',
    'zone_information', 'count_zone', 'count_item', 'move_item',
    'move_zone', 'resize_room', 'resize_zone', 'update_position'
]

# 2. Filter dataset agar intent 'add_stock' & 'remove_stock' dibuang
df_filtered = df[df['intent'].isin(intent_resmi)].dropna(subset=['text', 'intent']).copy()

# 3. Ambil data teks (X) dan label intent (y)
X = df_filtered['text'].astype(str).values
y = df_filtered['intent'].values

# 4. Ubah label string menjadi angka (Label Encoding)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)
class_names = label_encoder.classes_

# 5. Bagi data menjadi 80% untuk Training dan 20% untuk Validasi
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print("="*40)
print("HASIL FILTER DAN SPLITTING DATA:")
print("="*40)
print(f"Total data resmi digunakan : {len(df_filtered)} baris (Legacy berhasil dibuang!)")
print(f"Jumlah data training (80%) : {len(X_train)} baris")
print(f"Jumlah data validasi (20%): {len(X_val)} baris")
print(f"Total kelas intent unik    : {num_classes}")

HASIL FILTER DAN SPLITTING DATA:
Total data resmi digunakan : 2506 baris (Legacy berhasil dibuang!)
Jumlah data training (80%) : 2004 baris
Jumlah data validasi (20%): 502 baris
Total kelas intent unik    : 12


# **Tokenisasi dan Padding**

In [4]:
import json
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Konfigurasi parameter Tokenizer sesuai dengan karakteristik dataset baru
vocab_size = 2500        # Batas jumlah kosakata unik dalam kamus kata
max_length = 32          # Panjang maksimal kata dalam satu kalimat perintah
embedding_dim = 64       # Dimensi vektor untuk representasi makna kata
trunc_type = 'post'
padding_type = 'post'
oov_tok = "<OOV>"        # Token untuk menangani kata asing di luar kamus

# 2. Membuat dan melatih objek Tokenizer pada data latihan
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(X_train)

# 3. Mengonversi data teks (kalimat) menjadi urutan angka (sequence)
train_sequences = tokenizer.texts_to_sequences(X_train)
train_padded = pad_sequences(train_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

val_sequences = tokenizer.texts_to_sequences(X_val)
val_padded = pad_sequences(val_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

print("="*50)
print("HASIL TOKENISASI & PADDING SUKSES!")
print("="*50)
print(f"Bentuk matriks tensor data training : {train_padded.shape}")
print(f"Bentuk matriks tensor data validasi : {val_padded.shape}")
print("\nContoh kalimat asli dari dataset:")
print(f"  '{X_train[0]}'")
print("Hasil konversi menjadi representasi angka (Padding):")
print(f"  {train_padded[0]}")

HASIL TOKENISASI & PADDING SUKSES!
Bentuk matriks tensor data training : (2004, 32)
Bentuk matriks tensor data validasi : (502, 32)

Contoh kalimat asli dari dataset:
  'deskripsi lantai 2'
Hasil konversi menjadi representasi angka (Padding):
  [226   5   8   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0]


# **Membuat Custom Callback**

In [8]:
import tensorflow as tf

# Membuat komponen custom callback untuk memonitor akurasi validasi
class BestAccuracyCallback(tf.keras.callbacks.Callback):
    def __init__(self, target_accuracy=0.85):
        super(BestAccuracyCallback, self).__init__()
        self.target_accuracy = target_accuracy
        self.best_val_acc = 0.0

    def on_epoch_end(self, epoch, logs=None):
        val_acc = logs.get('val_accuracy')
        if val_acc is not None:
            # Jika akurasi validasi di epoch ini lebih tinggi dari yang terbaik sebelumnya
            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc
                self.model.save_weights('best_inventory_weights.weights.h5')
                print(f"\n[Callback] Epoch {epoch+1}: Akurasi validasi meningkat menjadi {val_acc:.4f}. Bobot model berhasil disimpan!")

            # Memberikan notifikasi jika performa model sudah melewati ambang kelulusan minimal
            if val_acc >= self.target_accuracy and val_acc == self.best_val_acc:
                print(f"[Callback] Target batas kelulusan minimal ({self.target_accuracy*100}%) aman terlampaui!")

# **Menyusun Arsitektur Model BiLSTM (Functional API)**

In [11]:
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.models import Model

# 1. Input Layer: Menerima urutan angka sepanjang 32 token (max_length)
input_layer = Input(shape=(max_length,), dtype='int32', name='input_teks')

# 2. Embedding Layer: Mengubah angka token menjadi vektor padat berdimensi 64 (embedding_dim)
embedding_layer = Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length)(input_layer)

# 3. Bidirectional LSTM: Membaca konteks kalimat maju-mundur secara bersamaan dengan 64 unit memory
bilstm_layer = Bidirectional(LSTM(64, return_sequences=False))(embedding_layer)

# 4. Dropout Layer: Memutus 40% koneksi acak saat latihan untuk mencegah over-fitting
dropout_layer = Dropout(0.4)(bilstm_layer)

# 5. Output Layer: Menggunakan Softmax untuk mengeluarkan skor probabilitas bagi 12 kelas intent
output_layer = Dense(num_classes, activation='softmax', name='output_intent')(dropout_layer)

# 6. Menyatukan semua layer ke dalam satu kesatuan Model Functional API
model = Model(inputs=input_layer, outputs=output_layer)

# 7. Compile Model: Menentukan fungsi kerugian (Loss) dan pengoptimal (Optimizer Adam)
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Menampilkan ringkasan arsitektur model di layar
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_teks (InputLayer)         │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, 32, 64)         │       160,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_intent (Dense)           │ (None, 12)             │         1,548 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 227,596 (889.05 KB)

 Trainable params: 227,596 (889.05 KB)

 Non-trainable params: 0 (0.00 B)

# **Training Model**

In [12]:
# 1. Mengaktifkan kustom callback (Target kelulusan minimal 85%)
accuracy_monitor = BestAccuracyCallback(target_accuracy=0.85)

# 2. Memulai proses training model selama 40 epoch
print("Memulai training model BiLSTM...")
history = model.fit(
    train_padded,
    y_train,
    epochs=40,
    batch_size=32,
    validation_data=(val_padded, y_val),
    callbacks=[accuracy_monitor]
)

# 3. Memanggil ekstensi file .weights.h5 yang sesuai dengan Keras 3
model.load_weights('best_inventory_weights.weights.h5')
print("\n" + "="*60)
print("[SUKSES TOTAL] Latihan selesai! Model menggunakan bobot terbaik.")
print("="*60)

Memulai training model BiLSTM...
Epoch 1/40
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.4494 - loss: 2.0561
[Callback] Epoch 1: Akurasi validasi meningkat menjadi 0.6315. Bobot model berhasil disimpan!
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 44ms/step - accuracy: 0.4775 - loss: 1.8116 - val_accuracy: 0.6315 - val_loss: 1.3291
Epoch 2/40
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.6027 - loss: 1.2569
[Callback] Epoch 2: Akurasi validasi meningkat menjadi 0.6892. Bobot model berhasil disimpan!
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 35ms/step - accuracy: 0.6347 - loss: 1.1829 - val_accuracy: 0.6892 - val_loss: 1.0288
Epoch 3/40
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.7057 - loss: 0.9197
[Callback] Epoch 3: Akurasi validasi meningkat menjadi 0.7530. Bobot model berhasil disimpan!
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.7221 - loss: 0.8369 - val_accuracy: 0.7530 - val_loss: 0.6862
Epoch 4/40
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.7980 - loss: 0.60

# **Evaluasi Akhir & Ekspor Berkas**

In [13]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

print("Mengunci model menggunakan konfigurasi performa terbaik...")

# 1. Mengunci bobot menggunakan nama file yang sesuai dengan Keras 3
model.load_weights('best_inventory_weights.weights.h5')
print("[BERHASIL] Bobot paling optimal telah berhasil diterapkan ke model!")

# 2. Menyimpan model utama ke format berkas .keras
model.save('inventory_classifier.keras')
print("[BERHASIL] File 'inventory_classifier.keras' siap diunduh.")

# 3. Mengekstrak dan menyimpan objek Tokenizer ke format .json
tokenizer_json = tokenizer.to_json()
with open('tokenizer.json', 'w', encoding='utf-8') as f:
    f.write(json.dumps(tokenizer_json, ensure_ascii=False))
print("[BERHASIL] File 'tokenizer.json' berhasil diekstrak.")

# 4. Menyusun Laporan Klasifikasi (Classification Report) ke dalam file teks
predictions = model.predict(val_padded, verbose=0)
y_pred = np.argmax(predictions, axis=1)
report = classification_report(y_val, y_pred, target_names=class_names)
with open('model_report.txt', 'w') as f:
    f.write(report)
print("[BERHASIL] Berkas ringkasan performa 'model_report.txt' berhasil disusun.")

# 5. Merender dan menyimpan Grafik Riwayat Training (Accuracy & Loss)
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Akurasi Latih', color='blue')
plt.plot(history.history['val_accuracy'], label='Akurasi Validasi', color='orange')
plt.title('Grafik Akurasi Model BiLSTM')
plt.xlabel('Epoch')
plt.ylabel('Akurasi')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Loss Latih', color='blue')
plt.plot(history.history['val_loss'], label='Loss Validasi', color='orange')
plt.title('Grafik Loss Model BiLSTM')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.savefig('training_history.png')
plt.close()
print("[BERHASIL] Grafik perkembangan 'training_history.png' berhasil disimpan.")

# 6. Merender Heatmap Confusion Matrix
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.title('Matriks Kebingungan (Confusion Matrix) Klasifikasi 12 Intent')
plt.xlabel('Prediksi Model')
plt.ylabel('Kenyataan Asli (Actual)')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.close()
print("[BERHASIL] Heatmap 'confusion_matrix.png' berhasil dirender.")

print("\n" + "="*50)
print("SEMUA BERKAS SIAP DIUNDUH!")
print("="*50)

Mengunci model menggunakan konfigurasi performa terbaik...
[BERHASIL] Bobot paling optimal telah berhasil diterapkan ke model!
[BERHASIL] File 'inventory_classifier.keras' siap diunduh.
[BERHASIL] File 'tokenizer.json' berhasil diekstrak.
[BERHASIL] Berkas ringkasan performa 'model_report.txt' berhasil disusun.
[BERHASIL] Grafik perkembangan 'training_history.png' berhasil disimpan.
[BERHASIL] Heatmap 'confusion_matrix.png' berhasil dirender.

SEMUA BERKAS SIAP DIUNDUH!


In [14]:
import json

print("="*50)
print("MEMULAI PROSES EKSPOR CONVERTER UNTUK BACKEND")
print("="*50)

# EXPORT TOKENIZER
tokenizer_json = tokenizer.to_json()
with open("tokenizer.json", "w", encoding="utf-8") as f:
    f.write(tokenizer_json)
print("Berkas 'tokenizer.json' berhasil dibuat!")

# EXPORT CLASS NAMES (Mengonversi ndarray menjadi list, lalu disimpan)
class_names_list = list(label_encoder.classes_)
with open("class_names.json", "w", encoding="utf-8") as f:
    json.dump(class_names_list, f, ensure_ascii=False, indent=2)

print("Berkas 'class_names.json' berhasil dibuat!")

print("\n" + "="*50)
print("BERKAS SIAP DIUNDUH!")
print("="*50)
print(f"Isi dari class_names.json kamu:\n{class_names_list}")

MEMULAI PROSES EKSPOR CONVERTER UNTUK BACKEND
Berkas 'tokenizer.json' berhasil dibuat!
Berkas 'class_names.json' berhasil dibuat!

BERKAS SIAP DIUNDUH!
Isi dari class_names.json kamu:
['check_stock', 'count_item', 'count_zone', 'low_stock', 'move_item', 'move_zone', 'resize_room', 'resize_zone', 'room_information', 'search_item', 'update_position', 'zone_information']
